# 00 - Collect Current Public User Ratings

This notebook now runs the user-interaction collector in two explicit phases. First it discovers public MAL usernames from clubs, recommendations, reviews, and forum pages, storing them only in a local encrypted retry queue. Then it consumes that queue through the official MAL API and writes anonymized ratings/profile features.

The key is stored locally under `secrets/`, and successful, low-signal, or likely duplicate users are removed from the pending queue. Failed users stay encrypted so the API pass can be retried later.

In [ ]:
from pathlib import Path
import json
import subprocess
import sys
import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

SCRIPT = ROOT / "src" / "00_collect_current_user_ratings.py"
QUEUE_FILE = ROOT / "data" / "build" / "current_usernames_encrypted_queue.json"
SUMMARY_FILE = ROOT / "data" / "build" / "current_user_ratings_summary.json"
RATINGS_FILE = ROOT / "data" / "processed" / "current_user_ratings.csv"
PROFILE_FILE = ROOT / "data" / "processed" / "current_user_profile_features.csv"

print(f"Project root: {ROOT}")
print(f"Collector script: {SCRIPT}")


Project root: C:\Users\CHAMPUX\Downloads\UPC TRABAJOS 2026\BIG DATA\proyect
Collector script: C:\Users\CHAMPUX\Downloads\UPC TRABAJOS 2026\BIG DATA\proyect\src\00_collect_current_user_ratings.py


## Run Helper

This helper streams script output directly under the cell. The long-running cells below are intentionally guarded by `RUN_* = False` so reopening the notebook never starts scraping by accident.

In [2]:
def run_stream(command):
    print(" ".join(str(part) for part in command))
    process = subprocess.Popen(
        [str(part) for part in command],
        cwd=ROOT,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    for line in process.stdout:
        print(line, end="")
    code = process.wait()
    if code != 0:
        raise RuntimeError(f"Command failed with exit code {code}")
    return code


## Status

This cell is safe: it only prints paths and current queue/output counts.

In [3]:
run_stream([sys.executable, SCRIPT])

queue = json.loads(QUEUE_FILE.read_text(encoding="utf-8")) if QUEUE_FILE.exists() else {"pending": {}, "completed_hashes": {}, "rejected_hashes": {}}
print("pending encrypted usernames:", len(queue.get("pending", {})))
print("completed hashes:", len(queue.get("completed_hashes", {})))
print("rejected hashes:", len(queue.get("rejected_hashes", {})))

if RATINGS_FILE.exists():
    print("rating rows:", sum(1 for _ in RATINGS_FILE.open("r", encoding="utf-8")) - 1)
if PROFILE_FILE.exists():
    print("profile rows:", sum(1 for _ in PROFILE_FILE.open("r", encoding="utf-8")) - 1)


c:\Users\CHAMPUX\AppData\Local\Programs\Python\Python313\python.exe C:\Users\CHAMPUX\Downloads\UPC TRABAJOS 2026\BIG DATA\proyect\src\00_collect_current_user_ratings.py
Encrypted username queue: data\build\current_usernames_encrypted_queue.json
Ratings output: data\processed\current_user_ratings.csv
Profile output: data\processed\current_user_profile_features.csv
Use --discover-users, --collect-ratings, or --run.
pending encrypted usernames: 358126
completed hashes: 41558
rejected hashes: 8743
rating rows: 11022162
profile rows: 41548


## Phase 1 - Discover Public Usernames

Discovery scans clubs first, then recent recommendations, reviews, and the configured forum boards/subboards. Usernames are encrypted immediately. The plaintext exists only in memory during discovery.

In [4]:
RUN_DISCOVERY = False

if RUN_DISCOVERY:
    run_stream([
        sys.executable, SCRIPT,
        "--discover-users",
        "--max-usernames", "1000000",
        "--source-order", "clubs", "recommendations", "reviews", "forums",
    ])
else:
    print("Set RUN_DISCOVERY = True to gather encrypted username candidates.")


Set RUN_DISCOVERY = True to gather encrypted username candidates.


## Phase 2 - Collect Ratings From Queue

Collection decrypts one queued username at a time, asks the official MAL API for scored list rows, writes anonymized rows for anime present in the catalog, then deletes the username from the pending queue on success. Users with 10 or fewer matched scored anime are rejected as too sparse. Failed users remain encrypted for retry.

In [5]:
RUN_COLLECTION = True
USER_LIMIT_THIS_RUN = None  # set None for an open-ended run

if RUN_COLLECTION:
    command = [sys.executable, SCRIPT, "--collect-ratings", "--min-ratings-per-user", "11"]
    if USER_LIMIT_THIS_RUN is not None:
        command += ["--user-limit", str(USER_LIMIT_THIS_RUN)]
    run_stream(command)
else:
    print("Set RUN_COLLECTION = True to consume the encrypted username queue.")


c:\Users\CHAMPUX\AppData\Local\Programs\Python\Python313\python.exe C:\Users\CHAMPUX\Downloads\UPC TRABAJOS 2026\BIG DATA\proyect\src\00_collect_current_user_ratings.py --collect-ratings --min-ratings-per-user 11
Loaded catalog_ids=15,492; known_voice_actor_person_ids=7,461
fetching queued user hash=fdf6b2a30f userID=1354665449 source=clubs
  failed 3/3; removed from queue: MAL auth/access error HTTP 403. Public list may be private.
fetching queued user hash=f059a764dc userID=1808730885 source=clubs
  failed 3/3; removed from queue: MAL auth/access error HTTP 403. Public list may be private.
fetching queued user hash=e41115af23 userID=247575050 source=clubs
  failed 3/3; removed from queue: MAL auth/access error HTTP 403. Public list may be private.
fetching queued user hash=5ada15144c userID=1344555202 source=clubs
  failed 3/3; removed from queue: MAL auth/access error HTTP 403. Public list may be private.
fetching queued user hash=b6305ca103 userID=654576709 source=clubs
  failed 3/

KeyboardInterrupt: 

## Output Preview

Use this after either phase to verify the queue and the anonymized outputs.

In [ ]:
if SUMMARY_FILE.exists():
    display(json.loads(SUMMARY_FILE.read_text(encoding="utf-8")))

if PROFILE_FILE.exists():
    display(pd.read_csv(PROFILE_FILE).tail(10))
else:
    print("No profile output yet.")

if RATINGS_FILE.exists():
    display(pd.read_csv(RATINGS_FILE).tail(10))
else:
    print("No ratings output yet.")
